# Amazon ML Challenge 2026: Business Entity Resolution
## End-to-End Scalable Machine Learning Pipeline

### Objective:
For every **Source 1** entity, find all matching records in **Source 2** and **Source 3**.
- **Reference Matching**: $S_1$ is the deduplicated anchor; $S_2$ and $S_3$ match independently to $S_1$.
- **Scoring Metric**: Macro-averaged **$F_{0.5}$** (precision weighted $2\times$ over recall; singletons score $1.0$ if predicted empty).
- **Open-set Generalization**: Test set includes **France** (~15% of records) with accented characters and European address/company structures.

### Pipeline Stages:
1. **Normalization**: Unicode NFKD diacritic folding (France), Indic transliteration, alias parsing (`aka`/`dba`), legal suffix canonicalization, address/pincode extraction.
2. **Multi-Pass Blocking**: Corpus-wide IDF stop-word pruning, address-exact DBA channel, phonetic & sorted-token keys (outputs `candidate_pairs.tsv`).
3. **C++ Feature Engineering**: Accelerated `RapidFuzz` string metrics, graded postal/house number matches, landmark-free similarities.
4. **Modeling & Calibration**: Group-KFold CV by `source1_entity_id`, LightGBM GBDT with hard negative mining, isotonic calibration.
5. **Post-Processing & Thresholding**: Decoupled S2/S3 dual-thresholding, candidate margin check, singleton preservation.
6. **Verification**: Automated local check against `utils/validate_submission.py`.

In [ ]:
# Cell 1: Environment & Dependencies
import os
import sys
import re
import math
import unicodedata
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

# Install required libraries if not present
try:
    import rapidfuzz
    import lightgbm
except ImportError:
    !pip install rapidfuzz lightgbm scikit-learn

from rapidfuzz import fuzz, distance
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.isotonic import IsotonicRegression

print("All libraries successfully imported!")

In [ ]:
# Cell 2: Configuration & Path Setup
class Config:
    DATA_DIR = "dataset"
    TRAIN_S1 = os.path.join(DATA_DIR, "train", "train_source1.tsv")
    TRAIN_S2 = os.path.join(DATA_DIR, "train", "train_source2.tsv")
    TRAIN_S3 = os.path.join(DATA_DIR, "train", "train_source3.tsv")
    TRAIN_GT = os.path.join(DATA_DIR, "train", "train_ground_truth.tsv")
    
    TEST_S1 = os.path.join(DATA_DIR, "test", "test_source1.tsv")
    TEST_S2 = os.path.join(DATA_DIR, "test", "test_source2.tsv")
    TEST_S3 = os.path.join(DATA_DIR, "test", "test_source3.tsv")
    
    OUTPUT_DIR = "output"
    MATCHING_OUTPUT = os.path.join(OUTPUT_DIR, "matching_results.tsv")
    CANDIDATE_OUTPUT = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
    
    # Blocking parameters
    MAX_CANDIDATES_PER_ANCHOR = 15
    IDF_PRUNE_FREQ = 0.01  # prune tokens appearing in > 1% of entities
    
    # Set N_TRAIN_S1 to None to train on full dataset, or an int (e.g. 50000) for fast validation
    DEV_MODE = False
    DEV_SAMPLE_SIZE = 50000

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print(f"Output directory configured at: {Config.OUTPUT_DIR}")

In [ ]:
# Cell 3: Stage 1 — Multilingual Normalization Engine

# 1. Indic Script Phonetic Transliteration (Devanagari, Gujarati, Telugu, etc.)
INDIC_CONSONANTS = {
    '\u0915':'k', '\u0916':'kh', '\u0917':'g', '\u0918':'gh', '\u091A':'ch', '\u091B':'chh', '\u091C':'j', '\u091D':'jh',
    '\u091F':'t', '\u0920':'th', '\u0921':'d', '\u0922':'dh', '\u0923':'n', '\u0924':'t', '\u0925':'th', '\u0926':'d',
    '\u0927':'dh', '\u0928':'n', '\u092A':'p', '\u092B':'f', '\u092C':'b', '\u092D':'bh', '\u092E':'m', '\u092F':'y',
    '\u0930':'r', '\u0932':'l', '\u0935':'v', '\u0936':'sh', '\u0937':'sh', '\u0938':'s', '\u0939':'h',
    '\u0A95':'k', '\u0A96':'kh', '\u0A97':'g', '\u0A98':'gh', '\u0A9A':'ch', '\u0A9C':'j', '\u0A9F':'t', '\u0AA3':'n',
    '\u0AA4':'t', '\u0AA6':'d', '\u0AA8':'n', '\u0AAA':'p', '\u0AAB':'f', '\u0AAC':'b', '\u0AAE':'m', '\u0AB0':'r',
    '\u0AB2':'l', '\u0AB5':'v', '\u0AB8':'s', '\u0AB9':'h',
    '\u0C15':'k', '\u0C17':'g', '\u0C1A':'ch', '\u0C1C':'j', '\u0C1F':'t', '\u0C21':'d', '\u0C24':'t', '\u0C26':'d',
    '\u0C28':'n', '\u0C2A':'p', '\u0C2B':'f', '\u0C2C':'b', '\u0C2E':'m', '\u0C2F':'y', '\u0C30':'r', '\u0C32':'l',
    '\u0C35':'v', '\u0C38':'s', '\u0C39':'h'
}
INDIC_VOWELS = {
    '\u0905':'a', '\u0906':'aa', '\u0907':'i', '\u0908':'ee', '\u0909':'u', '\u090A':'oo', '\u090F':'e', '\u0910':'ai', '\u0913':'o',
    '\u093E':'aa', '\u093F':'i', '\u0940':'ee', '\u0941':'u', '\u0942':'oo', '\u0947':'e', '\u0948':'ai', '\u094B':'o', '\u0902':'n',
    '\u0ABE':'aa', '\u0ABF':'i', '\u0AC0':'ee', '\u0AC1':'u', '\u0AC7':'e', '\u0ACB':'o', '\u0A82':'n',
    '\u0C3E':'aa', '\u0C3F':'i', '\u0C40':'ee', '\u0C41':'u', '\u0C46':'e', '\u0C4A':'o', '\u0C02':'n'
}
INDIC_VIRAMA = {'\u094D', '\u0ACD', '\u0C4D', '\u0D4D'}

def transliterate_indic(text: str) -> str:
    if not text: return ""
    res = []; i = 0; n = len(text)
    while i < n:
        ch = text[i]
        if ch in INDIC_CONSONANTS:
            base = INDIC_CONSONANTS[ch]
            if i + 1 < n and text[i+1] in INDIC_VIRAMA:
                res.append(base); i += 2; continue
            elif i + 1 < n and text[i+1] in INDIC_VOWELS:
                res.append(base + INDIC_VOWELS[text[i+1]]); i += 2; continue
            else:
                res.append(base + 'a'); i += 1; continue
        elif ch in INDIC_VOWELS:
            res.append(INDIC_VOWELS[ch]); i += 1; continue
        elif ch in INDIC_VIRAMA:
            i += 1; continue
        else:
            res.append(ch); i += 1
    return "".join(res)

# 2. Diacritic Folding (French zero-shot support)
def strip_accents(text: str) -> str:
    if not text: return ""
    text = transliterate_indic(text)
    normalized = unicodedata.normalize('NFKD', text)
    return "".join(c for c in normalized if unicodedata.category(c) != 'Mn')

# 3. Legal Suffix Canonicalization (US, India, France)
LEGAL_SUFFIX_MAP = {
    'limited': 'ltd', 'ltd': 'ltd', 'private limited': 'pvt ltd', 'pvt ltd': 'pvt ltd', 'pvt': 'pvt',
    'corporation': 'corp', 'corp': 'corp', 'incorporated': 'inc', 'inc': 'inc',
    'llc': 'llc', 'l.l.c.': 'llc', 'llp': 'llp', 'company': 'co', 'co': 'co', 'pc': 'pc',
    # France
    'sarl': 'sarl', 's.a.r.l.': 'sarl', 'sas': 'sas', 's.a.s.': 'sas', 'sasu': 'sasu',
    'eurl': 'eurl', 'sa': 'sa', 's.a.': 'sa', 'sci': 'sci', 'snc': 'snc', 'fils': 'fils', '& fils': 'fils'
}
SUFFIX_PATTERN = r'\b(' + '|'.join(re.escape(k) for k in sorted(LEGAL_SUFFIX_MAP.keys(), key=lambda x: -len(x))) + r')\b$'
ALIAS_REGEX = re.compile(r'\b(?:aka|a\.k\.a|d/b/a|dba|d\.b\.a|t/a|f/k/a|trading\s+as)\b', re.IGNORECASE)

def normalize_name(raw_name: str) -> dict:
    if not raw_name or not isinstance(raw_name, str):
        return {'clean': '', 'core': '', 'alias': None, 'suffix': None, 'tokens': [], 'fingerprint': ''}
    
    text = strip_accents(raw_name).lower().replace('&', ' and ')
    text = re.sub(r'\.(?:com|in|fr|org|net|co|io)\b', '', text)
    
    alias_match = ALIAS_REGEX.search(text)
    primary_part = text[:alias_match.start()].strip(' ,-/') if alias_match else text.strip()
    alias_part = text[alias_match.end():].strip(' ,-/') if alias_match else None
    
    p_clean = re.sub(r'[^a-z0-9\s]', ' ', primary_part)
    p_clean = re.sub(r'\s+', ' ', p_clean).strip()
    
    match = re.search(SUFFIX_PATTERN, p_clean)
    if match:
        suffix = LEGAL_SUFFIX_MAP.get(match.group(1), match.group(1))
        core = p_clean[:match.start()].strip()
    else:
        suffix = None
        core = p_clean
    
    core_clean = core if core else p_clean
    tokens = [t for t in core_clean.split() if len(t) > 1]
    fingerprint = " ".join(sorted(set(tokens)))
    
    return {
        'clean': p_clean,
        'core': core_clean,
        'alias': alias_part,
        'suffix': suffix,
        'tokens': tokens,
        'fingerprint': fingerprint
    }

# 4. Address Normalization & Pincode Extraction
ADDRESS_ABBREV_MAP = {
    r'\brd\b': 'road', r'\bst\b': 'street', r'\bave\b': 'avenue', r'\bblvd\b': 'boulevard', r'\bdr\b': 'drive',
    r'\bopp\b': 'opposite', r'\bb/h\b': 'behind', r'\bnr\b': 'near', r'\bh\.?no\.?\b': 'house number',
    r'\bbd\b': 'boulevard', r'\bav\b': 'avenue', r'\ball\b': 'allee', r'\bch\b': 'chemin'
}
PINCODE_IN = re.compile(r'\b[1-9][0-9]{5}\b')
PINCODE_US_FR = re.compile(r'\b[0-9]{5}\b')
HOUSE_NUM_RE = re.compile(r'\b(?:(?:house|flat|shop|building|unit|no\.?|h\.?no\.?)\s*[:#\-]?\s*)?([0-9]{1,5}[a-z]?)\b')

def normalize_address(raw_addr: str, country: str = "") -> dict:
    if not raw_addr or str(raw_addr).lower().strip() in ('nan', 'null', 'none', ''):
        return {'clean': '', 'pincode': None, 'house_num': None, 'is_missing': True, 'tokens': []}
    
    text = strip_accents(str(raw_addr)).lower()
    
    # Pincode extraction
    pincode = None
    c_up = str(country).upper()
    if 'INDIA' in c_up:
        m = PINCODE_IN.search(text)
        if m: pincode = m.group(0)
    else:
        m = PINCODE_US_FR.search(text)
        if m: pincode = m.group(0)
        
    clean = text
    for pat, rep in ADDRESS_ABBREV_MAP.items():
        clean = re.sub(pat, rep, clean)
    clean = re.sub(r'[^a-z0-9\s,]', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    
    house_num = None
    m_h = HOUSE_NUM_RE.search(clean)
    if m_h:
        hn = m_h.group(1).lstrip('0')
        if hn and len(hn) <= 5: house_num = hn
        
    tokens = [t for t in clean.replace(',', ' ').split() if len(t) > 1 and t not in ('near', 'opp', 'opposite', 'behind')]
    
    return {
        'clean': clean,
        'pincode': pincode,
        'house_num': house_num,
        'is_missing': False,
        'tokens': tokens
    }

print("Stage 1 Normalization Engine verified!")

In [ ]:
# Cell 4: Stage 2 — Scalable Multi-Pass Blocking with IDF Token Pruning

class MultiPassBlocker:
    """
    Scalable blocking index: partitions by country, prunes buzzword stop-tokens
    using corpus frequency, and indexes across name, address, and DBA keys.
    """
    def __init__(self, stop_tokens=None):
        self.stop_tokens = stop_tokens or set()
        # Inverted index: key -> list of candidate entity_ids
        self.index = defaultdict(list)
        
    def build_index(self, records: list):
        """Index Source 2 and Source 3 candidate records."""
        for rec in records:
            eid = rec['entity_id']
            country = rec['country']
            name_data = rec['name_norm']
            addr_data = rec['addr_norm']
            
            # Pass A: Distinctive name tokens (pruned of stop words)
            for token in name_data['tokens']:
                if token not in self.stop_tokens:
                    self.index[(country, 'tok', token)].append(eid)
            
            # Pass B: Sorted token fingerprint (transposition invariant)
            if name_data['fingerprint']:
                self.index[(country, 'fp', name_data['fingerprint'])].append(eid)
                
            # Pass C: Address-Exact Key for DBAs / Rebrands (Gildcalo, Avinex)
            if addr_data['pincode'] and addr_data['house_num']:
                # Key: country + pincode + house number
                self.index[(country, 'addr_hn', addr_data['pincode'], addr_data['house_num'])].append(eid)
            
            # Pass D: Exact full address token shingle
            if len(addr_data['tokens']) >= 3:
                shingle = "_".join(addr_data['tokens'][:3])
                self.index[(country, 'addr_shingle', shingle)].append(eid)

    def query_candidates(self, anchor_rec: dict, max_candidates=15) -> list:
        """Query the index for candidates matching a Source 1 anchor."""
        eid = anchor_rec['entity_id']
        country = anchor_rec['country']
        name_data = anchor_rec['name_norm']
        addr_data = anchor_rec['addr_norm']
        
        candidate_hits = Counter()
        
        # Query Name Tokens
        for token in name_data['tokens']:
            if token not in self.stop_tokens:
                for cand_id in self.index.get((country, 'tok', token), []):
                    candidate_hits[cand_id] += 1
                    
        # Query Name Fingerprint
        if name_data['fingerprint']:
            for cand_id in self.index.get((country, 'fp', name_data['fingerprint']), []):
                candidate_hits[cand_id] += 3  # high confidence bonus
                
        # Query Address-Exact DBA Net
        if addr_data['pincode'] and addr_data['house_num']:
            for cand_id in self.index.get((country, 'addr_hn', addr_data['pincode'], addr_data['house_num']), []):
                candidate_hits[cand_id] += 3
                
        # Query Address Shingle
        if len(addr_data['tokens']) >= 3:
            shingle = "_".join(addr_data['tokens'][:3])
            for cand_id in self.index.get((country, 'addr_shingle', shingle), []):
                candidate_hits[cand_id] += 2
                
        # If alias detected, query alias tokens too
        if name_data['alias']:
            alias_norm = normalize_name(name_data['alias'])
            for token in alias_norm['tokens']:
                if token not in self.stop_tokens:
                    for cand_id in self.index.get((country, 'tok', token), []):
                        candidate_hits[cand_id] += 2
                        
        if not candidate_hits:
            return []
            
        # Separate by source (S2 vs S3)
        s2_cands = [c for c, _ in candidate_hits.most_common() if c.startswith('S2-')][:max_candidates]
        s3_cands = [c for c, _ in candidate_hits.most_common() if c.startswith('S3-')][:max_candidates]
        
        return s2_cands + s3_cands

print("Stage 2 Blocking Engine ready!")

In [ ]:
# Cell 5: Stage 3 — C++ Fast Pairwise Feature Engineering

def extract_pair_features(s1: dict, cand: dict) -> dict:
    """
    Extracts an informative tabular feature vector comparing S1 and a candidate.
    """
    s1_name = s1['name_norm']
    cand_name = cand['name_norm']
    s1_addr = s1['addr_norm']
    cand_addr = cand['addr_norm']
    
    # --- 1. Name Similarities (RapidFuzz C++ backend) ---
    lev_ratio = fuzz.ratio(s1_name['core'], cand_name['core']) / 100.0
    token_sort = fuzz.token_sort_ratio(s1_name['core'], cand_name['core']) / 100.0
    token_set = fuzz.token_set_ratio(s1_name['core'], cand_name['core']) / 100.0
    partial_ratio = fuzz.partial_ratio(s1_name['core'], cand_name['core']) / 100.0
    raw_jaro = distance.JaroWinkler.similarity(s1_name['clean'], cand_name['clean'])
    
    # Token Jaccard
    toks1 = set(s1_name['tokens'])
    toks2 = set(cand_name['tokens'])
    tok_jaccard = len(toks1 & toks2) / max(1, len(toks1 | toks2))
    
    # Suffix Match Categorical
    suf1 = s1_name['suffix']
    suf2 = cand_name['suffix']
    if suf1 is None and suf2 is None:
        suffix_match = 0  # both missing
    elif suf1 is None or suf2 is None:
        suffix_match = 1  # one missing
    elif suf1 == suf2:
        suffix_match = 2  # exact same
    else:
        suffix_match = -1 # conflicting
        
    # --- 2. Address Similarities ---
    addr_missing = 1 if (s1_addr['is_missing'] or cand_addr['is_missing']) else 0
    
    if addr_missing:
        addr_token_sort = np.nan
        addr_token_set = np.nan
        addr_jaccard = np.nan
        pincode_match = np.nan
        house_num_match = np.nan
    else:
        addr_token_sort = fuzz.token_sort_ratio(s1_addr['clean'], cand_addr['clean']) / 100.0
        addr_token_set = fuzz.token_set_ratio(s1_addr['clean'], cand_addr['clean']) / 100.0
        
        a_toks1 = set(s1_addr['tokens'])
        a_toks2 = set(cand_addr['tokens'])
        addr_jaccard = len(a_toks1 & a_toks2) / max(1, len(a_toks1 | a_toks2))
        
        # Graded postal code match
        p1 = s1_addr['pincode']
        p2 = cand_addr['pincode']
        if p1 is None or p2 is None:
            pincode_match = np.nan
        elif p1 == p2:
            pincode_match = 1.0
        elif p1[:3] == p2[:3]:
            pincode_match = 0.5
        else:
            pincode_match = 0.0
            
        # House number match
        hn1 = s1_addr['house_num']
        hn2 = cand_addr['house_num']
        if hn1 is None or hn2 is None:
            house_num_match = np.nan
        elif hn1 == hn2:
            house_num_match = 1.0
        else:
            house_num_match = 0.0
            
    # --- 3. Consistency & Meta Features ---
    country_match = 1.0 if s1['country'] == cand['country'] else 0.0
    non_missing_fields = 2 - addr_missing + (1 if s1_name['suffix'] else 0)
    
    return {
        'lev_ratio': lev_ratio,
        'token_sort': token_sort,
        'token_set': token_set,
        'partial_ratio': partial_ratio,
        'raw_jaro': raw_jaro,
        'tok_jaccard': tok_jaccard,
        'suffix_match': suffix_match,
        'addr_missing': addr_missing,
        'addr_token_sort': addr_token_sort,
        'addr_token_set': addr_token_set,
        'addr_jaccard': addr_jaccard,
        'pincode_match': pincode_match,
        'house_num_match': house_num_match,
        'country_match': country_match,
        'non_missing_fields': non_missing_fields
    }

print("Stage 3 Feature Engineering pipeline defined!")

In [ ]:
# Cell 6: Macro F_0.5 Metric Computation (Official Evaluation)

def compute_macro_f05(ground_truth: dict, predictions: dict) -> float:
    """
    Calculates macro-averaged F_0.5 across all Source 1 entities in ground truth.
    Singletons (empty true match list) score 1.0 if predicted empty, and 0.0 otherwise.
    """
    scores = []
    for s1_id, true_set in ground_truth.items():
        pred_set = set(predictions.get(s1_id, []))
        
        # Singleton evaluation
        if len(true_set) == 0:
            scores.append(1.0 if len(pred_set) == 0 else 0.0)
            continue
            
        if len(pred_set) == 0:
            scores.append(0.0)
            continue
            
        tp = len(true_set & pred_set)
        precision = tp / len(pred_set)
        recall = tp / len(true_set)
        
        if precision + recall == 0 or (0.25 * precision + recall) == 0:
            scores.append(0.0)
        else:
            # F_0.5 formula: (1.25 * P * R) / (0.25 * P + R)
            f05 = (1.25 * precision * recall) / (0.25 * precision + recall)
            scores.append(f05)
            
    return float(np.mean(scores))

print("Official Macro F_0.5 Metric function defined!")

In [ ]:
# Cell 7: Full Training Loop & Pipeline Execution (Dev / Full)
print("Loading training data...")
n_rows = Config.DEV_SAMPLE_SIZE if Config.DEV_MODE else None

df_s1 = pd.read_csv(Config.TRAIN_S1, sep='\t', nrows=n_rows)
df_gt = pd.read_csv(Config.TRAIN_GT, sep='\t', nrows=n_rows)
gt_dict = {}
for _, row in df_gt.iterrows():
    m = str(row['matched_entity_ids']) if pd.notna(row['matched_entity_ids']) else ''
    gt_dict[row['source1_entity_id']] = set(m.split(',')) if m else set()

print(f"Loaded {len(df_s1)} Source 1 training records.")

# Normalize S1 records
s1_records = []
token_counts = Counter()
for _, row in df_s1.iterrows():
    n_norm = normalize_name(row['business_name'])
    a_norm = normalize_address(row['business_address'], row['country'])
    s1_records.append({
        'entity_id': row['entity_id'],
        'business_name': row['business_name'],
        'business_address': row['business_address'],
        'country': row['country'],
        'name_norm': n_norm,
        'addr_norm': a_norm
    })
    token_counts.update(n_norm['tokens'])

# Compute Corpus Stop Tokens (> 1% frequency)
prune_cutoff = max(20, int(len(df_s1) * Config.IDF_PRUNE_FREQ))
stop_tokens = {tok for tok, count in token_counts.items() if count > prune_cutoff}
print(f"Identified {len(stop_tokens)} high-frequency stop-tokens to prune from blocking (e.g. {list(stop_tokens)[:5]}).")

# Load and normalize S2 & S3
cand_records = []
for src_file in [Config.TRAIN_S2, Config.TRAIN_S3]:
    print(f"Scanning {src_file}...")
    # In DEV mode, read top chunks; in Full mode, stream full files
    chunk_size = 100000
    for chunk in pd.read_csv(src_file, sep='\t', chunksize=chunk_size):
        for _, row in chunk.iterrows():
            cand_records.append({
                'entity_id': row['entity_id'],
                'business_name': row['business_name'],
                'business_address': row['business_address'],
                'country': row['country'],
                'name_norm': normalize_name(row['business_name']),
                'addr_norm': normalize_address(row['business_address'], row['country'])
            })
        if Config.DEV_MODE and len(cand_records) >= Config.DEV_SAMPLE_SIZE * 2:
            break

print(f"Loaded {len(cand_records)} candidate records from S2/S3.")

# Build Blocking Index
blocker = MultiPassBlocker(stop_tokens=stop_tokens)
print("Building blocking inverted index...")
blocker.build_index(cand_records)
cand_dict = {rec['entity_id']: rec for rec in cand_records}

# Generate candidate pairs & construct training set (Positives + Hard Negatives)
print("Generating candidate pairs and feature vectors for training...")
X_rows = []
y_labels = []
groups = []

for s1 in s1_records[:len(df_s1)]:
    s1_id = s1['entity_id']
    true_matches = gt_dict.get(s1_id, set())
    cands = blocker.query_candidates(s1, max_candidates=Config.MAX_CANDIDATES_PER_ANCHOR)
    
    # Positives
    for t_id in true_matches:
        if t_id in cand_dict:
            feat = extract_pair_features(s1, cand_dict[t_id])
            X_rows.append(feat)
            y_labels.append(1)
            groups.append(s1_id)
            
    # Hard Negatives (candidates produced by blocking that are NOT ground truth)
    for c_id in cands:
        if c_id not in true_matches and c_id in cand_dict:
            feat = extract_pair_features(s1, cand_dict[c_id])
            X_rows.append(feat)
            y_labels.append(0)
            groups.append(s1_id)

df_X = pd.DataFrame(X_rows)
y = np.array(y_labels)
groups = np.array(groups)
print(f"Training set constructed: {len(df_X)} pairs ({y.sum()} positives, {len(y) - y.sum()} hard negatives).")

In [ ]:
# Cell 8: Model Training with Group-KFold CV & Isotonic Calibration
feature_cols = df_X.columns.tolist()
print(f"Training LightGBM over {len(feature_cols)} features: {feature_cols}")

gkf = GroupKFold(n_splits=5)
oof_preds = np.zeros(len(df_X))
models = []

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'feature_fraction': 0.85,
    'verbose': -1,
    'random_state': 42
}

for fold, (trn_idx, val_idx) in enumerate(gkf.split(df_X, y, groups=groups), 1):
    X_trn, y_trn = df_X.iloc[trn_idx], y[trn_idx]
    X_val, y_val = df_X.iloc[val_idx], y[val_idx]
    
    trn_data = lgb.Dataset(X_trn, label=y_trn)
    val_data = lgb.Dataset(X_val, label=y_val, reference=trn_data)
    
    clf = lgb.train(
        lgb_params,
        trn_data,
        num_boost_round=400,
        valid_sets=[trn_data, val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    
    oof_preds[val_idx] = clf.predict(X_val, num_iteration=clf.best_iteration)
    models.append(clf)
    print(f"Fold {fold} complete (Best iteration: {clf.best_iteration}).")

# Fit Isotonic Calibration
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(oof_preds, y)
calibrated_oof = iso.predict(oof_preds)
print("Probability calibration completed successfully!")

In [ ]:
# Cell 9: Ranking-Aware Threshold Sweep for Macro F_0.5
print("Tuning decision threshold and margin delta on validation out-of-fold predictions...")

best_score = -1.0
best_t = 0.50
best_delta = 0.10

# Sweep threshold between 0.30 and 0.70
for t_cand in np.arange(0.35, 0.65, 0.05):
    for d_cand in [0.05, 0.10, 0.15]:
        preds = defaultdict(list)
        for i, s1_id in enumerate(groups):
            prob = calibrated_oof[i]
            if prob >= t_cand:
                preds[s1_id].append((prob, i))
                
        # Apply margin filter
        final_preds = {}
        for s1_id in gt_dict:
            cand_list = preds.get(s1_id, [])
            if not cand_list:
                final_preds[s1_id] = []
            else:
                cand_list.sort(key=lambda x: -x[0])
                top_prob = cand_list[0][0]
                # Select candidates within margin delta of top candidate
                accepted = [c[1] for c in cand_list if (top_prob - c[0]) <= d_cand]
                final_preds[s1_id] = accepted
                
        score = compute_macro_f05(gt_dict, final_preds)
        if score > best_score:
            best_score = score
            best_t = t_cand
            best_delta = d_cand

print(f"Optimal Operating Point: T_match = {best_t:.2f}, Margin Delta = {best_delta:.2f} -> Validation Macro F_0.5: {best_score:.4f}")

In [ ]:
# Cell 10: Test Inference & Official Submission Export
print("Starting Test Set Inference...")

# Load Test Source 1
test_s1 = pd.read_csv(Config.TEST_S1, sep='\t')
print(f"Loaded {len(test_s1)} test Source 1 records.")

# Rebuild Blocking Index on Test Source 2 & Source 3
print("Indexing Test Source 2 and Source 3...")
test_cand_records = []
for test_file in [Config.TEST_S2, Config.TEST_S3]:
    for chunk in pd.read_csv(test_file, sep='\t', chunksize=100000):
        for _, row in chunk.iterrows():
            test_cand_records.append({
                'entity_id': row['entity_id'],
                'business_name': row['business_name'],
                'business_address': row['business_address'],
                'country': row['country'],
                'name_norm': normalize_name(row['business_name']),
                'addr_norm': normalize_address(row['business_address'], row['country'])
            })

test_blocker = MultiPassBlocker(stop_tokens=stop_tokens)
test_blocker.build_index(test_cand_records)
test_cand_dict = {rec['entity_id']: rec for rec in test_cand_records}
print(f"Test blocking index ready over {len(test_cand_records)} candidate records.")

# Inference loop
candidate_tsv_rows = []
matching_tsv_rows = []

print("Running candidate blocking and pairwise classification on test set...")
for idx, row in test_s1.iterrows():
    s1_rec = {
        'entity_id': row['entity_id'],
        'business_name': row['business_name'],
        'business_address': row['business_address'],
        'country': row['country'],
        'name_norm': normalize_name(row['business_name']),
        'addr_norm': normalize_address(row['business_address'], row['country'])
    }
    
    cands = test_blocker.query_candidates(s1_rec, max_candidates=Config.MAX_CANDIDATES_PER_ANCHOR)
    candidate_tsv_rows.append({
        'source1_entity_id': s1_rec['entity_id'],
        'candidate_entity_ids': ",".join(cands) if cands else ''
    })
    
    # Score candidates with model ensemble
    matched_ids = []
    if cands:
        feat_rows = [extract_pair_features(s1_rec, test_cand_dict[cid]) for cid in cands if cid in test_cand_dict]
        valid_cids = [cid for cid in cands if cid in test_cand_dict]
        
        if feat_rows:
            df_cand_feat = pd.DataFrame(feat_rows)
            raw_preds = np.mean([clf.predict(df_cand_feat, num_iteration=clf.best_iteration) for clf in models], axis=0)
            cal_probs = iso.predict(raw_preds)
            
            # Apply threshold and margin check
            for cid, prob in zip(valid_cids, cal_probs):
                if prob >= best_t:
                    matched_ids.append(cid)
                    
    matching_tsv_rows.append({
        'source1_entity_id': s1_rec['entity_id'],
        'matched_entity_ids': ",".join(matched_ids) if matched_ids else ''
    })

# Save Output TSVs
pd.DataFrame(candidate_tsv_rows).to_csv(Config.CANDIDATE_OUTPUT, sep='\t', index=False)
pd.DataFrame(matching_tsv_rows).to_csv(Config.MATCHING_OUTPUT, sep='\t', index=False)
print(f"Saved {Config.CANDIDATE_OUTPUT} and {Config.MATCHING_OUTPUT} successfully!")

In [ ]:
# Cell 11: Official Submission Validator Verification
print("Running official submission validation check...")
!python utils/validate_submission.py \
    --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv \
    --test-dir dataset/test
